# BrainScan-v2 — Test du classifieur InceptionV3 pré-entraîné

Ce notebook charge `models/classification/best_model.keras` et teste une prédiction sur une image, puis peut évaluer le modèle sur le split de test préparé localement.

- Modèle : InceptionV3 + tête de classification, format `tf_keras`.
- Entrée : image RGB redimensionnée en `299 × 299`, pixels ramenés dans `[0, 1]`.
- Classes (ordre alphabétique des dossiers) : `glioma`, `meningioma`, `notumor`, `pituitary`.

Cette démonstration est destinée à l'expérimentation et ne constitue pas un outil de diagnostic médical.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import tf_keras as keras
from PIL import Image

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "notebooks").is_dir() and (candidate / "models").is_dir():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable. Ouvrez ce notebook depuis le dépôt BrainScan-v2.")

PROJECT_ROOT = find_project_root()
MODEL_PATH = PROJECT_ROOT / "models" / "classification" / "best_model.keras"
DATA_DIR = PROJECT_ROOT / "data" / "Data"
PROCESSED_TEST_DIR = PROJECT_ROOT / "data" / "processed" / "classification" / "test"
IMAGE_SIZE = (299, 299)

print(f"Projet : {PROJECT_ROOT}")
print(f"Modèle : {MODEL_PATH}")
print(f"TensorFlow : {tf.__version__} / tf_keras : {keras.__version__}")

## Charger et vérifier le modèle

Le modèle utilise une couche `tensorflow_hub.KerasLayer`. Au premier chargement, TensorFlow Hub peut télécharger son module InceptionV3 depuis Internet.

In [ ]:
if not MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"Modèle introuvable : {MODEL_PATH}\n"
        "Téléchargez-le en suivant les instructions de la section « Modèle pré-entraîné » du ReadMe.md."
    )

model = keras.models.load_model(
    MODEL_PATH,
    custom_objects={"KerasLayer": hub.KerasLayer},
    compile=False,
)

if tuple(model.input_shape[1:]) != (*IMAGE_SIZE, 3):
    raise ValueError(f"Taille d'entrée inattendue : {model.input_shape}")
if model.output_shape[-1] != 4:
    raise ValueError(f"Le modèle devrait produire 4 classes, sortie obtenue : {model.output_shape}")

model.summary()

## Prédiction sur une image

Modifiez `IMAGE_PATH` pour tester une autre image. L'image d'exemple est choisie dans le dossier `glioma` du jeu de données brut.

In [ ]:
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Jeu de données absent : {DATA_DIR}")

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
example_images = sorted(
    path for path in (DATA_DIR / CLASS_NAMES[0]).iterdir()
    if path.is_file() and path.suffix.lower() in image_extensions
)
if not example_images:
    raise FileNotFoundError(f"Aucune image trouvée dans {DATA_DIR / CLASS_NAMES[0]}")

IMAGE_PATH = example_images[0]  # Remplacez cette valeur pour tester votre propre image.
with Image.open(IMAGE_PATH) as image_file:
    image = image_file.convert("RGB").resize(IMAGE_SIZE, Image.Resampling.BILINEAR)

image_array = np.asarray(image, dtype=np.float32) / 255.0
probabilities = model.predict(image_array[None, ...], verbose=0)[0]
predicted_index = int(np.argmax(probabilities))

print(f"Image : {IMAGE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Classe prédite : {CLASS_NAMES[predicted_index]} ({probabilities[predicted_index]:.1%})")
for class_name, probability in zip(CLASS_NAMES, probabilities, strict=True):
    print(f"  {class_name:<12} {probability:.1%}")

plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.title(f"Prédiction : {CLASS_NAMES[predicted_index]}")
plt.axis("off");

## Évaluation facultative sur le split de test

Cette cellule utilise `data/processed/classification/test/` s'il existe. Elle ne s'exécute pas sur le jeu brut pour éviter de présenter des images d'entraînement comme un test indépendant.

In [ ]:
if PROCESSED_TEST_DIR.is_dir():
    test_class_names = sorted(path.name for path in PROCESSED_TEST_DIR.iterdir() if path.is_dir())
    if test_class_names != CLASS_NAMES:
        raise ValueError(
            f"Classes du split de test {test_class_names} différentes des classes du modèle {CLASS_NAMES}"
        )

    test_ds = keras.utils.image_dataset_from_directory(
        PROCESSED_TEST_DIR,
        labels="inferred",
        label_mode="int",
        class_names=CLASS_NAMES,
        color_mode="rgb",
        batch_size=16,
        image_size=IMAGE_SIZE,
        shuffle=False,
        interpolation="bilinear",
    )
    test_ds = test_ds.map(
        lambda images, labels: (tf.cast(images, tf.float32) / 255.0, labels),
        num_parallel_calls=1,
    ).prefetch(1)

    y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
    y_prob = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(y_prob, axis=1)

    from sklearn.metrics import ConfusionMatrixDisplay, classification_report

    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        display_labels=CLASS_NAMES,
        xticks_rotation=45,
        cmap="Blues",
    )
    plt.tight_layout()
else:
    print(f"Split de test absent ({PROCESSED_TEST_DIR}); exécutez d'abord notebooks/2-preparation-split.ipynb.")